## ALGORITHM 3: DECISION TREE

### Definition
Decision Tree is a **flowchart-like structure** where each internal node represents a test on a feature, each branch represents the outcome, and each leaf node represents a class label. It mimics human decision-making through a series of IF-ELSE conditions.

### Indian Analogy: Arranged Marriage Decision 💒
```
                    [Is Boy/Girl Employed?]
                         /          \
                       Yes           No
                       /              \
              [Salary > 10LPA?]    REJECT ❌
                 /        \
               Yes         No
               /            \
      [Same Caste?]    [Family Business?]
         /    \            /      \
       Yes    No         Yes      No
        |      |          |        |
    ACCEPT   [Check    ACCEPT   REJECT
      ✅     Kundli]     ✅       ❌

    Evaluation Metrics for Decision Trees
    Same as Logistic Regression (Classification):

    Accuracy, Precision, Recall, F1, AUC-ROC

    Plus additional metrics:

    Feature Importance: Which features matter most?
    Tree Depth: How complex is the model?
    Pruning: Prevent overfitting by limiting tree growth

In [1]:
# ============================================================
# DECISION TREE - COMPLETE IMPLEMENTATION WITH EVALUATION
# ============================================================

# Step 1: Import Libraries
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Step 2: Create Dataset - Credit Card Fraud Detection
np.random.seed(42)
n_transactions = 1000

data = {
    'amount': np.random.exponential(5000, n_transactions),
    'hour_of_day': np.random.randint(0, 24, n_transactions),
    'is_weekend': np.random.choice([0, 1], n_transactions),
    'distance_from_home': np.random.exponential(50, n_transactions),
    'online_transaction': np.random.choice([0, 1], n_transactions),
    'transaction_count_24h': np.random.randint(1, 20, n_transactions),
}
df = pd.DataFrame(data)

In [3]:
df

,amount,hour_of_day,is_weekend,distance_from_home,online_transaction,transaction_count_24h
0,2346.340450,14,0,48.501531,0,1
1,15050.607155,11,1,21.722169,0,9
2,6583.728468,15,0,92.083492,1,15
3,4564.712769,23,1,31.865492,1,1
4,848.124352,18,0,193.412121,1,12
...,...,...,...,...,...,...
995,480.253673,2,1,137.663327,0,13
996,12463.499218,19,1,3.307329,0,3
997,735.652244,22,0,87.075158,1,1
998,15002.453266,3,0,17.292622,1,4


In [4]:
# Fraud logic (unusual patterns)
fraud_score = (
    (df['amount'] > 10000).astype(int) +
    ((df['hour_of_day'] > 22) | (df['hour_of_day'] < 5)).astype(int) +
    (df['distance_from_home'] > 100).astype(int) +
    (df['transaction_count_24h'] > 10).astype(int)
)
df['is_fraud'] = (fraud_score >= 2).astype(int)

In [5]:
df

,amount,hour_of_day,is_weekend,distance_from_home,online_transaction,transaction_count_24h,is_fraud
0,2346.340450,14,0,48.501531,0,1,0
1,15050.607155,11,1,21.722169,0,9,0
2,6583.728468,15,0,92.083492,1,15,0
3,4564.712769,23,1,31.865492,1,1,0
4,848.124352,18,0,193.412121,1,12,1
...,...,...,...,...,...,...,...
995,480.253673,2,1,137.663327,0,13,1
996,12463.499218,19,1,3.307329,0,3,0
997,735.652244,22,0,87.075158,1,1,0
998,15002.453266,3,0,17.292622,1,4,1


In [6]:
# Add some noise
noise_idx = np.random.choice(n_transactions, size=int(n_transactions*0.03), replace=False)
df.loc[noise_idx, 'is_fraud'] = 1 - df.loc[noise_idx, 'is_fraud']

In [8]:
print("=== Credit Card Transaction Data ===")
print(df.head(10))
print(f"\nFraud Rate: {df['is_fraud'].mean()*100:.2f}%")

# Step 3: Prepare Data
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

=== Credit Card Transaction Data ===
         amount  hour_of_day  is_weekend  distance_from_home  \
0   2346.340450           14           0           48.501531   
1  15050.607155           11           1           21.722169   
2   6583.728468           15           0           92.083492   
3   4564.712769           23           1           31.865492   
4    848.124352           18           0          193.412121   
5    847.981460            7           0           50.275519   
6    299.193843           20           1            6.748893   
7  10056.154322           16           1           56.377997   
8   4595.410768           22           0           19.659835   
9   6156.250309            4           1           57.970243   

   online_transaction  transaction_count_24h  is_fraud  
0                   0                      1         0  
1                   0                      9         0  
2                   1                     15         0  
3                   1         

In [9]:
# Step 4: Train Basic Model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [10]:
# Step 5: Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)[:, 1]


In [13]:
# Step 6: EVALUATION
print(f"\n{'='*70}")
print(f"=== DECISION TREE EVALUATION ===")
print(f"{'='*70}")

# --- Basic Metrics ---
print(f"\n--- Performance Metrics ---")
print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred)*100:.2f}%")
print(f"Testing Accuracy: {accuracy_score(y_test, y_test_pred)*100:.2f}%")

# Check for Overfitting
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
if train_acc - test_acc > 0.1:
    print(f"\n⚠️ WARNING: Possible OVERFITTING!")
    print(f"   Training Accuracy ({train_acc:.2%}) >> Test Accuracy ({test_acc:.2%})")
    print(f"   Consider: Pruning, max_depth limit, min_samples_leaf")

# --- Detailed Metrics on Test Set ---
print(f"\n--- Test Set Detailed Metrics ---")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_test_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_test_prob):.4f}")

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_test_pred)
print(f"\n--- Confusion Matrix ---")
print(f"              Predicted")
print(f"            Legit   Fraud")
print(f"Actual Legit  {cm[0,0]:4d}    {cm[0,1]:4d}")
print(f"Actual Fraud  {cm[1,0]:4d}    {cm[1,1]:4d}")

# --- Tree Statistics ---
print(f"\n--- Tree Structure ---")
print(f"Tree Depth: {model.get_depth()}")
print(f"Number of Leaves: {model.get_n_leaves()}")
print(f"Total Nodes: {model.tree_.node_count}")

# --- Feature Importance ---
print(f"\n--- Feature Importance ---")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance.to_string(index=False))



=== DECISION TREE EVALUATION ===

--- Performance Metrics ---
Training Accuracy: 100.00%
Testing Accuracy: 92.00%

--- Test Set Detailed Metrics ---
Precision: 0.8393
Recall: 0.8704
F1 Score: 0.8545
AUC-ROC: 0.9044

--- Confusion Matrix ---
              Predicted
            Legit   Fraud
Actual Legit   137       9
Actual Fraud     7      47

--- Tree Structure ---
Tree Depth: 16
Number of Leaves: 59
Total Nodes: 117

--- Feature Importance ---
              Feature  Importance
transaction_count_24h    0.331723
          hour_of_day    0.226649
   distance_from_home    0.226370
               amount    0.209047
           is_weekend    0.004906
   online_transaction    0.001305


In [14]:
# Step 7: CROSS-VALIDATION
print(f"\n{'='*70}")
print(f"=== CROSS-VALIDATION (5-Fold) ===")
print(f"{'='*70}")

cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"\nFold Accuracies: {np.round(cv_scores, 4)}")
print(f"Mean Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


=== CROSS-VALIDATION (5-Fold) ===

Fold Accuracies: [0.94  0.95  0.91  0.92  0.905]
Mean Accuracy: 0.9250 ± 0.0173
